# Kayıp Fonksiyonları

Bu alıştırmada, Kayıp fonksiyonlarının `LinearRegression` modeli üzerindeki etkilerini karşılaştıracaksınız.

👇 Bu zorluk için kullanmak üzere bir CSV dosyası indirelim ve onu bir DataFrame'e dönüştürelim

In [1]:
import pandas as pd

data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/loss_functions_dataset.csv")
data.sample(5)

,Relative Compactness,Surface Area,Wall Area,Roof Area,Overall Height,Glazing Area,Average Temperature
269,0.71,710.5,269.5,220.5,3.5,0.10,12.225
117,0.76,661.5,416.5,122.5,7.0,0.10,33.235
311,0.76,661.5,416.5,122.5,7.0,0.25,36.525
735,0.82,612.5,318.5,147.0,7.0,0.40,31.230
317,0.71,710.5,269.5,220.5,3.5,0.25,14.250


🎯 Göreviniz, tasarımına göre bir seranın içindeki ortalama sıcaklığı tahmin etmektir. Sıcaklık tahminleriniz, her bir bitki için iklim ihtiyaçlarına göre uygun sera tasarımını seçmenize yardımcı olacaktır.

🌿 Bitkilerin küçük sıcaklık değişimlerini kaldırabildiğini, ancak sıcaklık değişimleri arttıkça katlanarak daha duyarlı hale geldiğini biliyorsunuz.

## 1. Teori

❓ Teorik olarak, bitkileri öldürme riskini sınırlamak için modelinizi hangi Kayıp fonksiyonu üzerinde eğitirsiniz?

<details>
<summary> 🆘 Cevap </summary>
    
Teorik olarak, Ortalama Kare Hata (MSE) Kayıp fonksiyonunu kullanırsınız. Bu, aykırı tahminleri cezalandırır ve modelinizin büyük hatalar yapmasını engeller. Bu, daha küçük sıcaklık değişimleri ve bitkiler için daha düşük risk sağlayacaktır.

</details>

Ortalama Kare Hata (MSE) kayıp fonksiyonunu kullanırdım. Çünkü büyük hataları daha fazla cezalandırır ve bu sayede modelin büyük sıcaklık sapmaları yapmasını engeller. Bu da bitkilerin zarar görme riskini azaltır.

## 2. Uygulama

### 2.1 Ön İşleme

❓ Özellikleri standartlaştırın

In [3]:
X = data.drop(columns=["Average Temperature"])
y = data["Average Temperature"]

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### 2.2 Modelleme

Bu bölümde, farklı Kayıp fonksiyonları üzerinde optimize edilmiş modelleri değerlendirerek teoriyi doğrulayacaksınız.

### En Küçük Kareler (MSE) Kaybı

❓ **En Küçük Kareler Kaybı** (MSE) üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

In [11]:
from sklearn.linear_model import SGDRegressor

model_mse = SGDRegressor(loss="squared_error", random_state=42)

❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru ve bunu `r2` değişkeninde kaydedin
- Tüm katlarınızın °C cinsinden en büyük tek tahmin hatasını hesaplayın ve `max_error_celsius` değişkeninde kaydedin

(İpucu: `max_error` sklearn'de kabul edilen bir puanlama metriğidir)

In [27]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, max_error

# R2 score
r2_scores = cross_val_score(model_mse, X_scaled, y, cv=10, scoring="r2")
r2 = r2_scores.mean()
r2

0.8982272875896993

In [28]:
# max_error scorer tanımlama
max_error_scorer = make_scorer(max_error, greater_is_better=False)

max_errors = cross_val_score(model_mse, X_scaled, y, cv=10, scoring=max_error_scorer)
max_error_celsius = -max_errors.mean()
max_error_celsius

8.739697100767762

### Ortalama Mutlak Hata (MAE) Kaybı

Peki modelimizi MAE üzerinde optimize edersek ne olur?

❓ **MAE** Kaybı üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

<details>
<summary>💡 İpuçları</summary>

- MAE kaybı `SGDRegressor`'da doğrudan belirtilemez. Doğru parametreleri ayarlayarak tasarlanması gerekir

</details>

In [14]:
model_mae = SGDRegressor(loss="epsilon_insensitive",random_state=42)

❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru, bunu `r2_mae`'de saklayın
- Tüm katlarınızın en büyük tek tahmin hatasını, bunu `max_error_mae`'de saklayın

In [25]:
r2_scores_mae = cross_val_score(model_mae, X_scaled, y, cv=10, scoring="r2")
r2_mae = r2_scores_mae.mean()
r2_mae

0.8755826687124214

In [26]:
max_error_scorer = make_scorer(max_error, greater_is_better=False)

max_errors_mae = cross_val_score(model_mae, X_scaled, y, cv=10, scoring=max_error_scorer)
max_error_mae = -max_errors_mae.mean()
max_error_mae

10.906698494705095

## 3. Sonuç

❓ Değerlendirdiğiniz modellerden hangisi göreviniz için en uygun görünüyor?

<details>
<summary> 🆘Cevap </summary>
    
İki model arasında ortalama çapraz doğrulanmış r2 skorları yaklaşık olarak benzer olmasına rağmen, MAE üzerinde optimize edilen modelin zaman zaman daha büyük hatalar yapma şansı daha fazladır, bu da bitkileri öldürme riskini artırır!
    
</details>

Her iki modelin ortalama R2 skorları benzer olsa da, MAE ile optimize edilen model zaman zaman daha büyük hatalar yapmaktadır. Bu durum, sera sıcaklığının kritik seviyelerde yanlış tahmin edilmesine ve bitkilerin zarar görmesine neden olabilir. Bu nedenle, büyük hataları daha fazla cezalandıran MSE kaybı bu görev için daha uygundur.

# 🏁 Kodunuzu kontrol edin ve notebook'unuzu gönderin

In [29]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'loss_functions',
    r2 = r2,
    r2_mae = r2_mae,
    max_error = max_error_celsius,
    max_error_mae = max_error_mae
)

result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/semih/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/semih/code/Semih0799/S16D1-S-data-knn/S16D4-S-loss-functions/tests
plugins: anyio-4.8.0, typeguard-4.4.2
collecting ... collected 3 items

test_loss_functions.py::TestLossFunctions::test_max_error_order PASSED   [ 33%]
test_loss_functions.py::TestLossFunctions::test_r2 PASSED                [ 66%]
test_loss_functions.py::TestLossFunctions::test_r2_mae PASSED            [100%]

============================== 3 passed in 0.27s ===============================


💯 You can commit your code:

git add tests/loss_functions.pickle

git commit -m 'Completed loss_functions step'

git push origin master

